# 6.2 Logistic Regression

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 6: Classification Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-06-classification-algorithms/06.02_logistic-regression.ipynb)

Companion code for Section 6.2 of the book. Run the cells in order from top to bottom.

**Example 6.2.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.datasets import load_breast_cancer

# ── Load data ─────────────────────────────────────────────────────
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      stratify=y, random_state=42)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)

# ── Logistic Regression from scratch ──────────────────────────────
class LogisticRegressionScratch:
    def __init__(self, lr=0.1, n_epochs=1000):
        self.lr = lr; self.n_epochs = n_epochs
        self.weights = None; self.bias = None; self.losses = []

    def sigmoid(self, z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        n, p = X.shape
        self.weights = np.zeros(p); self.bias = 0.0
        for _ in range(self.n_epochs):
            z    = X @ self.weights + self.bias
            p_hat= self.sigmoid(z)
            dw   = (1/n) * X.T @ (p_hat - y)
            db   = (1/n) * (p_hat - y).sum()
            self.weights -= self.lr * dw
            self.bias    -= self.lr * db
            loss = -np.mean(y*np.log(p_hat+1e-15) + (1-y)*np.log(1-p_hat+1e-15))
            self.losses.append(loss)
        return self

    def predict_proba(self, X):
        return self.sigmoid(X @ self.weights + self.bias)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def score(self, X, y):
        return (self.predict(X) == y).mean()

# Train from scratch
lr_scratch = LogisticRegressionScratch(lr=0.1, n_epochs=500)
lr_scratch.fit(X_tr, y_train)
print(f'From-scratch accuracy: {lr_scratch.score(X_te, y_test):.4f}')

# ── Sklearn Logistic Regression ───────────────────────────────────
models = {
    'No regularisation': LogisticRegression(C=1e10, max_iter=1000, random_state=42),
    'L2 (C=1.0)':        LogisticRegression(C=1.0,  penalty='l2', max_iter=1000, random_state=42),
    'L2 (C=0.01)':       LogisticRegression(C=0.01, penalty='l2', max_iter=1000, random_state=42),
    'L1 (C=1.0)':        LogisticRegression(C=1.0,  penalty='l1', solver='liblinear', random_state=42),
}

print(f'\n{"Model":<22} {"Train Acc":>10} {"Test Acc":>10} {"AUC":>8} {"Non-zero coef":>14}')
print('='*70)
for name, model in models.items():
    model.fit(X_tr, y_train)
    tr_acc  = model.score(X_tr, y_train)
    te_acc  = model.score(X_te, y_test)
    auc     = roc_auc_score(y_test, model.predict_proba(X_te)[:,1])
    nonzero = (np.abs(model.coef_[0]) > 1e-6).sum()
    print(f'{name:<22} {tr_acc:>10.4f} {te_acc:>10.4f} {auc:>8.4f} {nonzero:>14}/{X.shape[1]}')

# ── Detailed report for best model ────────────────────────────────
best = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
best.fit(X_tr, y_train)
y_pred = best.predict(X_te)
print('\nDetailed Classification Report:')
print(classification_report(y_test, y_pred, target_names=data.target_names))

# ── Coefficient interpretation ────────────────────────────────────
coef_df = pd.DataFrame({
    'feature':    data.feature_names,
    'coefficient':best.coef_[0],
    'odds_ratio': np.exp(best.coef_[0])  # e^β = odds ratio
}).sort_values('coefficient', key=abs, ascending=False)
print('\nTop 5 most influential features:')
print(coef_df.head(5).round(4).to_string(index=False))
print('\nOdds ratio interpretation: coef=0.5 → odds of benign multiply by e^0.5=1.65')

# ── Visualisation: Decision boundary (2D projection) ─────────────
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_tr)
X_2d_te = pca.transform(X_te)

lr_2d = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_2d.fit(X_2d, y_train)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Logistic Regression Analysis', fontsize=14, fontweight='bold')

# Decision boundary
ax = axes[0]
h = 0.1
xx, yy = np.meshgrid(np.arange(X_2d[:,0].min()-1, X_2d[:,0].max()+1, h),
                     np.arange(X_2d[:,1].min()-1, X_2d[:,1].max()+1, h))
Z = lr_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
colors = ['#E53935' if c==0 else '#1976D2' for c in y_train]
ax.scatter(X_2d[:,0], X_2d[:,1], c=colors, s=20, alpha=0.7, edgecolors='white', lw=0.5)
ax.set_title('Decision Boundary (PCA 2D)'); ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

# Loss convergence
ax = axes[1]
ax.plot(lr_scratch.losses, 'b-', lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Binary Cross-Entropy Loss')
ax.set_title('Training Loss Convergence'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('logistic_regression.png', dpi=150, bbox_inches='tight')
plt.show()